# 04 — Backtest SMACross

Run our SMACross strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + SMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweep — fast/slow grid with heatmap

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.indicators import SimpleMovingAverage
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.core import bar_type_str
from src.strategies.sma_cross import SMACross, SMACrossConfig

from charts import plot_sma_cross

# ── Shared config ────────────────────────────────────────────────
CATALOG_PATH    = "../data/catalog"
INSTRUMENT_ID   = "BTC-USD-PERP.HYPERLIQUID"
BAR_TYPE_STR    = bar_type_str(INSTRUMENT_ID, "1h")
VENUE           = Venue("HYPERLIQUID")
STARTING_CAPITAL = 10_000
TRADE_SIZE      = Decimal("0.01")   # 0.01 BTC per trade (~$650 notional at $65k)
FAST_SMA        = 20
SLOW_SMA        = 50
FAST_PERIODS = sorted(set([5, 10, 15, 20, 100] + [FAST_SMA]))
SLOW_PERIODS = sorted(set([30, 50, 75, 100, 200] + [SLOW_SMA]))

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

print(f"Instrument : {instrument.id}")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = BacktestEngine(config=BacktestEngineConfig(
    logging=LoggingConfig(log_level="ERROR"),
))

engine.add_venue(
    venue=VENUE,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    base_currency=None,              # Multi-currency — settlement from instrument
    starting_balances=[Money(STARTING_CAPITAL, USDC)],
)

engine.add_instrument(instrument)
engine.add_data(bars)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add SMACross strategy + run ───────────────────────────
config = SMACrossConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=TRADE_SIZE,
    fast_sma_period=FAST_SMA,
    slow_sma_period=SLOW_SMA,
)
strategy = SMACross(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: Price chart with SMA overlay + trade markers ──────────

fig = plot_sma_cross(
    bars, fills_report,
    fast_period=FAST_SMA,
    slow_period=SLOW_SMA,
    instrument_label=INSTRUMENT_ID,
    bar_label="1h",
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 8: Equity curve ──────────────────────────────────────────
#
# The analyzer's returns() gives per-period returns (populated by
# calculate_statistics above). Fallback to account balance if empty.

fig, ax = plt.subplots(figsize=(14, 5))
plotted = False

# Method 1: Cumulative returns from analyzer
try:
    returns = analyzer.returns()
    if returns is not None and len(returns) > 0:
        cumulative = (1 + returns).cumprod()
        cumulative.plot(ax=ax, label="Cumulative Return")
        plotted = True
except Exception:
    pass

# Method 2: Fallback to account balance curve
if not plotted and account_report is not None and not account_report.empty:
    account_report.plot(ax=ax, label="Account Balance")
    ax.set_ylabel("Balance (USDC)")
    plotted = True

if plotted:
    ax.set_title(f"Equity Curve — SMACross({FAST_SMA}/{SLOW_SMA})  BTC 1h", fontsize=13)
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No returns or account data available for equity curve.")

In [ ]:
# ── Cell 9: Summary statistics ────────────────────────────────────
#
# calculate_statistics() was already called above (Cell 6).

general_stats = analyzer.get_performance_stats_general()
pnl_stats     = analyzer.get_performance_stats_pnls(USDC)
returns_stats  = analyzer.get_performance_stats_returns()

print("=== General ===")
for k, v in general_stats.items():
    print(f"  {k}: {v}")

print("\n=== PnL (USDC) ===")
for k, v in pnl_stats.items():
    print(f"  {k}: {v}")

print("\n=== Returns ===")
for k, v in returns_stats.items():
    print(f"  {k}: {v}")

print(f"\nTotal PnL      : {analyzer.total_pnl(USDC)}")
print(f"Total PnL %    : {analyzer.total_pnl_percentage(USDC)}")
print(f"Positions      : {len(positions)}")

In [ ]:
# ── Cell 10: Parameter sweep ──────────────────────────────────────
#
# Grid search over fast/slow SMA period combinations.
#

def make_engine():
    """Create a fresh BacktestEngine with venue, instrument, and data."""
    eng = BacktestEngine(config=BacktestEngineConfig(
        logging=LoggingConfig(log_level="ERROR"),
    ))
    eng.add_venue(
        venue=VENUE,
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        base_currency=None,
        starting_balances=[Money(STARTING_CAPITAL, USDC)],
    )
    eng.add_instrument(instrument)
    eng.add_data(bars)
    return eng

def run_single_backtest(fast: int, slow: int) -> dict:
    """Fresh engine per combo — no shared state."""
    eng = make_engine()

    cfg = SMACrossConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_size=TRADE_SIZE,
        fast_sma_period=fast,
        slow_sma_period=slow,
    )
    eng.add_strategy(SMACross(cfg))

    row = {"fast": fast, "slow": slow}

    try:
        eng.run()

        a = eng.portfolio.analyzer
        acct = eng.cache.account_for_venue(VENUE)
        pos = eng.cache.position_snapshots() + eng.cache.positions()

        if acct is None:
            row["error"] = "no account"
            row.update(total_pnl=np.nan, total_pnl_pct=np.nan,
                       num_positions=len(pos), final_balance=np.nan,
                       min_balance=np.nan)
        else:
            a.calculate_statistics(acct, pos)
            balance = float(acct.balance_total(USDC))
            row.update(
                total_pnl=float(a.total_pnl(USDC)),
                total_pnl_pct=float(a.total_pnl_percentage(USDC)),
                num_positions=len(pos),
                final_balance=balance,
                error="",
            )

            # Detect if equity ever hit zero during the run
            acct_report = eng.trader.generate_account_report(VENUE)
            if not acct_report.empty:
                min_bal = acct_report["total"].astype(float).min()
                row["min_balance"] = min_bal
                if min_bal <= 0:
                    row["error"] = "liquidated"
            else:
                row["min_balance"] = balance

            try:
                for k, v in a.get_performance_stats_general().items():
                    row[k] = v
            except Exception as e:
                print(f"  Warning: general stats failed for {fast}/{slow}: {e}")
            try:
                for k, v in a.get_performance_stats_pnls(USDC).items():
                    row[k] = v
            except Exception as e:
                print(f"  Warning: PnL stats failed for {fast}/{slow}: {e}")
            try:
                for k, v in a.get_performance_stats_returns().items():
                    row[k] = v
            except Exception as e:
                print(f"  Warning: returns stats failed for {fast}/{slow}: {e}")
    except Exception as e:
        row["error"] = str(e)
        row.update(total_pnl=np.nan, total_pnl_pct=np.nan,
                   num_positions=0, final_balance=np.nan, min_balance=np.nan)
    finally:
        eng.dispose()

    return row

# ── Run the sweep ──
results = []
combos = [(f, s) for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
total = len(combos)

for i, (fast, slow) in enumerate(combos, 1):
    row = run_single_backtest(fast, slow)
    results.append(row)
    status = f"  !! {row['error']}" if row.get("error") else ""
    print(
        f"  [{i}/{total}] fast={fast:>3}, slow={slow:>3}  "
        f"PnL={row['total_pnl']:>10.2f} PnL%={row['total_pnl_pct']:>7.2f}%"
        f"  positions={row['num_positions']}{status}"
    )

results_df = pd.DataFrame(results)

# Show key columns (some stat keys may vary — show what's available)
display_cols = ["fast", "slow", "total_pnl", "total_pnl_pct",
                "num_positions", "final_balance", "min_balance", "error"]
for col in results_df.columns:
    lower = col.lower()
    if any(kw in lower for kw in ["sharpe", "drawdown", "win", "trades", "expectancy", "factor"]):
        display_cols.append(col)

display_cols = [c for c in display_cols if c in results_df.columns]
print("\n=== Parameter Sweep Results ===")
display(results_df[display_cols])

In [ ]:
# ── Cell 11: PnL heatmap ──────────────────────────────────────────
pivot = results_df.pivot(index="slow", columns="fast", values="total_pnl")

fig, ax = plt.subplots(figsize=(8, 6))

# Diverging colourmap centred at 0
vmax = max(abs(np.nanmin(pivot.values)), abs(np.nanmax(pivot.values)))
im = ax.imshow(
    pivot.values,
    cmap="RdYlGn",
    aspect="auto",
    vmin=-vmax,
    vmax=vmax,
)

# Labels
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index])
ax.set_xlabel("Fast SMA Period")
ax.set_ylabel("Slow SMA Period")
ax.set_title("Total PnL (USDC) by SMA Parameters")

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if np.isnan(val):
            continue
        color = "white" if abs(val) > vmax * 0.6 else "black"
        ax.text(j, i, f"{val:,.0f}", ha="center", va="center", fontsize=10, color=color)

fig.colorbar(im, ax=ax, label="PnL (USDC)")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 12: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")